# NexaMem Quick Start with Azure Redis - Basic Authentication

This notebook demonstrates how to use **NexaMem** (Next-Generation AI Memory) with **Azure Cache for Redis** using **Access Key authentication** for straightforward setup and testing.

## What is NexaMem?

NexaMem is a Python package for managing AI chatbot conversation history with:
- Multiple storage backends (memory, file, SQLite, Redis)
- Channel-based memory organization
- Advanced features like encryption, quotas, and TTL
- Azure Cloud integration

## Prerequisites

Before running this notebook, ensure you have:
1. An **Azure Cache for Redis** instance
2. **Access keys** from Azure Portal (Primary or Secondary key)
3. **Network access** configured (firewall rules, if needed)

## When to Use This Approach

**Access Key authentication** is ideal for:
- 🚀 **Getting started** quickly with NexaMem
- 🧪 **Development and testing** environments
- 🔧 **Proof of concepts** and prototypes
- 📚 **Learning** how NexaMem works with Azure Redis

For production environments, consider the [Enterprise notebook](quickstart_azure_redis_enterprise.ipynb) with Entra ID authentication.

## 1. Install Required Dependencies

First, let's install the necessary packages:

In [1]:
# Install NexaMem and Redis dependencies
!pip install nexamem redis

print("✅ Installation completed!")

✅ Installation completed!


## 2. Import Required Modules

In [2]:
import os
import sys
import json
from typing import Dict, Any

# Add local src directory to path for development version
repo_root = os.getcwd()
src_path = os.path.join(repo_root, 'src')
if os.path.exists(src_path):
    sys.path.insert(0, src_path)
    
    # Force reload of modules if they're already loaded
    modules_to_reload = ['nexamem', 'nexamem.redis_adapter', 'nexamem.history', 'nexamem.aimemory']
    for module in modules_to_reload:
        if module in sys.modules:
            del sys.modules[module]

# NexaMem imports
from nexamem import initialize, ChatHistory, AIMemory, MemoryScope
from nexamem.redis_adapter import AzureRedisConnectionHelper

print("✅ All imports successful!")
if 'nexamem' in sys.modules:
    print(f"📍 Using NexaMem from: {sys.modules['nexamem'].__file__}")

✅ All imports successful!
📍 Using NexaMem from: c:\Users\chadada\source\repos\nexamem\src\nexamem\__init__.py


## 3. Azure Redis Configuration with Access Keys

### 🔑 Get Your Azure Redis Details

1. Go to [Azure Portal](https://portal.azure.com)
2. Navigate to your **Azure Cache for Redis** instance
3. In the left menu, click **"Access keys"**
4. Copy the **Primary connection string** or **Access keys**
5. Note your **Redis hostname** (e.g., `myredis.redis.cache.windows.net`)

### Option A: Direct Configuration (Replace Placeholders)

In [3]:
# Azure Redis Configuration with Access Key
# 🚨 TODO: Replace these placeholders with your actual Azure Redis details

AZURE_REDIS_CONFIG = {
    "history_storage": "redis",
    "redis_config": {
        # 🔄 Replace with your Azure Redis hostname
        "azure_hostname": "YOUR-REDIS-NAME.redis.cache.windows.net",
        
        # 🔑 Replace with your primary or secondary access key from Azure Portal
        "azure_access_key": "YOUR-REDIS-ACCESS-KEY-HERE",
        
        # Optional: Database number (default: 0)
        "db": 0,
        
        # Optional: Custom key prefix for organization
        "key_prefix": "nexamem:basic",
        
        # Optional: Connection timeout settings
        "socket_timeout": 5.0,
        "socket_connect_timeout": 5.0
    },
    "debug": True
}

print("📋 Configuration prepared (remember to update placeholders!)")
print(f"   Hostname: {AZURE_REDIS_CONFIG['redis_config']['azure_hostname']}")
print(f"   Key prefix: {AZURE_REDIS_CONFIG['redis_config']['key_prefix']}")
print(f"   Database: {AZURE_REDIS_CONFIG['redis_config']['db']}")

📋 Configuration prepared (remember to update placeholders!)
   Hostname: YOUR-REDIS-NAME.redis.cache.windows.net
   Key prefix: nexamem:basic
   Database: 0


### Option B: Environment Variables (Recommended for Code Sharing)

In [ ]:
# Alternative: Environment-based configuration
# Set these environment variables before running:
# - AZURE_REDIS_HOSTNAME=your-redis.redis.cache.windows.net
# - AZURE_REDIS_ACCESS_KEY=your-access-key-here

def create_config_from_env():
    """Create Redis configuration from environment variables."""
    hostname = os.getenv('AZURE_REDIS_HOSTNAME')
    access_key = os.getenv('AZURE_REDIS_ACCESS_KEY')
    
    if not hostname or not access_key:
        print("❌ Missing environment variables:")
        print("   - AZURE_REDIS_HOSTNAME")
        print("   - AZURE_REDIS_ACCESS_KEY")
        print("\n💡 To set them (Windows):")
        print("   set AZURE_REDIS_HOSTNAME=your-redis.redis.cache.windows.net")
        print("   set AZURE_REDIS_ACCESS_KEY=your-access-key")
        print("\n💡 To set them (Linux/Mac):")
        print("   export AZURE_REDIS_HOSTNAME=your-redis.redis.cache.windows.net")
        print("   export AZURE_REDIS_ACCESS_KEY=your-access-key")
        return None
    
    return {
        "history_storage": "redis",
        "redis_config": {
            "azure_hostname": hostname,
            "azure_access_key": access_key,
            "db": 0,
            "key_prefix": "nexamem:env"
        },
        "debug": False
    }

ENV_CONFIG = create_config_from_env()
if ENV_CONFIG:
    print("✅ Environment-based configuration ready")
    print(f"   Hostname: {ENV_CONFIG['redis_config']['azure_hostname']}")
else:
    print("ℹ️  Environment variables not set - will use manual configuration")

## 4. Test Connection Configuration

Before initializing NexaMem, let's validate the Redis connection configuration:

In [ ]:
# Test Azure Redis connection configuration
try:
    # Use environment config if available, otherwise use manual config
    test_config = ENV_CONFIG if ENV_CONFIG else AZURE_REDIS_CONFIG
    
    # Extract connection details
    hostname = test_config['redis_config']['azure_hostname']
    access_key = test_config['redis_config']['azure_access_key']
    
    # Generate Redis configuration using the helper
    redis_config = AzureRedisConnectionHelper.get_azure_redis_config(
        hostname=hostname,
        access_key=access_key
    )
    
    print("✅ Redis configuration generated successfully:")
    for key, value in redis_config.items():
        if key == 'password':
            # Mask the password for security
            masked = f"{'*' * 8}...{value[-4:] if len(value) > 4 else '***'}"
            print(f"   {key}: {masked}")
        else:
            print(f"   {key}: {value}")
            
    print("\n🔗 Connection Details:")
    print(f"   Protocol: {'SSL' if redis_config['ssl'] else 'Plain'}")
    print(f"   Port: {redis_config['port']} (Azure Redis standard SSL port)")
    print(f"   Database: {test_config['redis_config']['db']}")
            
except Exception as e:
    print(f"❌ Configuration error: {e}")
    print("\n🔧 Make sure to:")
    print("   1. Replace placeholder values with actual Azure Redis details")
    print("   2. Verify hostname format: your-redis.redis.cache.windows.net")
    print("   3. Use primary or secondary access key from Azure Portal")

## 5. Initialize NexaMem with Azure Redis

Now let's initialize NexaMem with your Azure Redis configuration:

In [ ]:
# Choose which configuration to use
config_to_use = ENV_CONFIG if ENV_CONFIG else AZURE_REDIS_CONFIG

try:
    # Initialize NexaMem with Azure Redis
    initialize(config_to_use)
    print("🎉 NexaMem successfully initialized with Azure Redis!")
    
    # Test the connection
    history = ChatHistory()
    print("✅ ChatHistory instance created successfully")
    print("\n🔐 Using Access Key authentication")
    print(f"🏷️  Key prefix: {config_to_use['redis_config']['key_prefix']}")
    
except Exception as e:
    print(f"❌ Initialization failed: {e}")
    print("\n🔧 Troubleshooting checklist:")
    print("   1. ✓ Azure Redis hostname is correct")
    print("   2. ✓ Access key is valid and active")
    print("   3. ✓ Network connectivity to Azure (check firewall)")
    print("   4. ✓ Redis allows SSL connections on port 6380")
    print("   5. ✓ Your IP address is allowed by Azure Redis firewall")
    print("\n🌐 Test connectivity:")
    print("   telnet YOUR-REDIS-NAME.redis.cache.windows.net 6380")

## 6. Basic ChatHistory Usage

Let's demonstrate basic chatbot conversation management:

In [ ]:
# Set up session context
history = ChatHistory()

# Configure the conversation context
history.set_session_context(
    agent_id="basic_assistant",
    user_id="demo_user_123",
    session_id="basic_session",
    env="demo"
)

print("👤 Session context configured:")
print(f"   Agent: basic_assistant")
print(f"   User: demo_user_123")
print(f"   Session: basic_session")
print(f"   Environment: demo")
print(f"\n🔑 Authentication: Azure Redis Access Keys")

In [ ]:
# Add conversation messages
print("💬 Adding conversation messages...\n")

# User messages
history.add_user_message(
    "Hello! I'm testing NexaMem with Azure Redis using access keys.", 
    tokens=12
)

# Assistant responses
history.add_assistant_message(
    "Great! You're successfully connected to Azure Cache for Redis using access key authentication. "
    "This is the simplest way to get started with NexaMem and Azure Redis.",
    tokens=28
)

history.add_user_message(
    "What are the advantages of using access key authentication?",
    tokens=10
)

history.add_assistant_message(
    "Access key authentication offers several advantages:\n"
    "1. **Simple setup**: Just hostname and key needed\n"
    "2. **Quick testing**: Perfect for development and POCs\n"
    "3. **No identity complexity**: No Azure AD setup required\n"
    "4. **Universal compatibility**: Works with any Redis client\n"
    "5. **Immediate access**: No permissions or role assignments\n\n"
    "For production, consider Entra ID authentication for enhanced security.",
    tokens=68
)

print("✅ Messages added to Azure Redis successfully!")

In [ ]:
# Retrieve and display conversation history
messages = history.get_history()

print(f"📚 Retrieved {len(messages)} messages from Azure Redis:\n")

for i, msg in enumerate(messages, 1):
    role_icon = "👤" if msg['role'] == 'user' else "🤖"
    print(f"{i}. {role_icon} **{msg['role'].title()}** ({msg.get('tokens', 0)} tokens)")
    print(f"   {msg['content']}\n")

# Show total token usage
total_tokens = sum(msg.get('tokens', 0) for msg in messages)
print(f"📊 Total conversation: {total_tokens} tokens")

## 7. Connection Health Check

Let's verify the Redis connection is working properly:

In [ ]:
# Test Redis connection health
try:
    # Access the underlying Redis adapter for health checks
    from nexamem.config import get_config
    
    config = get_config()
    if hasattr(config, 'redis_adapter') and config.redis_adapter:
        adapter = config.redis_adapter
        
        # Ping test
        ping_result = adapter.ping()
        print(f"🏓 Redis ping: {'✅ Success' if ping_result else '❌ Failed'}")
        
        # Connection info
        info = adapter.get_info()
        print(f"\n🔗 Connection Information:")
        print(f"   Host: {info.get('host', 'unknown')}")
        print(f"   Port: {info.get('port', 'unknown')}")
        print(f"   SSL: {info.get('ssl', 'unknown')}")
        print(f"   Database: {info.get('db', 'unknown')}")
        print(f"   Azure Redis: {info.get('is_azure', 'unknown')}")
        print(f"   Connected: {info.get('connected', 'unknown')}")
        
        # Authentication method
        print(f"\n🔐 Authentication: Access Keys")
        print(f"   Simple and direct connection to Azure Redis")
        
    else:
        print("❌ No Redis adapter found in configuration")
        
except Exception as e:
    print(f"❌ Health check error: {e}")

## 8. Advanced Features Demo

Let's explore some advanced NexaMem features with Azure Redis:

In [ ]:
# Test different conversation contexts
print("🎭 Testing multiple conversation contexts...\n")

# Create separate sessions for different users/agents
contexts = [
    {"agent_id": "customer_support", "user_id": "customer_001", "session_id": "support_ticket_123"},
    {"agent_id": "sales_assistant", "user_id": "prospect_456", "session_id": "sales_call_789"},
    {"agent_id": "tech_advisor", "user_id": "user_789", "session_id": "tech_consultation_456"}
]

for i, ctx in enumerate(contexts, 1):
    # Create new history instance for each context
    ctx_history = ChatHistory()
    ctx_history.set_session_context(
        agent_id=ctx["agent_id"],
        user_id=ctx["user_id"],
        session_id=ctx["session_id"],
        env="demo"
    )
    
    # Add a test message
    ctx_history.add_user_message(f"Hello from {ctx['agent_id']}!", tokens=5)
    ctx_history.add_assistant_message(f"Welcome! This is {ctx['agent_id']} ready to help.", tokens=8)
    
    # Retrieve messages
    ctx_messages = ctx_history.get_history()
    
    print(f"{i}. 🎯 Context: {ctx['agent_id']} -> {ctx['user_id']}")
    print(f"   Session: {ctx['session_id']}")
    print(f"   Messages: {len(ctx_messages)}")
    print()

print("✅ All conversations stored independently in Azure Redis!")
print("\n🏗️  Each conversation context is isolated and secure.")

## 9. Data Cleanup

Let's clean up the demo data:

In [ ]:
# Optional: Clean up demo data
cleanup_confirmed = input("Do you want to clean up demo data? (y/N): ").lower().strip()

if cleanup_confirmed == 'y':
    try:
        # Clean up main demo session
        demo_scope = history._get_current_scope()
        
        # Use bulk_erase if available (from advanced features)
        from nexamem.config import get_config
        config = get_config()
        
        if hasattr(config, 'redis_adapter') and config.redis_adapter:
            adapter = config.redis_adapter
            if hasattr(adapter, 'bulk_erase'):
                deleted_count = adapter.bulk_erase(demo_scope)
                print(f"🗑️  Deleted {deleted_count} keys from main demo session")
            
            # Clean up additional contexts
            total_deleted = deleted_count
            for ctx in contexts:
                from nexamem.memory_scope import MemoryScope
                ctx_scope = MemoryScope(
                    agent_id=ctx["agent_id"],
                    user_id=ctx["user_id"],
                    session_id=ctx["session_id"],
                    env="demo"
                )
                if hasattr(adapter, 'bulk_erase'):
                    ctx_deleted = adapter.bulk_erase(ctx_scope)
                    total_deleted += ctx_deleted
            
            print(f"📊 Total cleanup: {total_deleted} keys deleted")
            print("✅ Demo data cleanup completed!")
        else:
            print("❌ Could not access Redis adapter for cleanup")
            
    except Exception as e:
        print(f"❌ Cleanup error: {e}")
        print("   Data will naturally expire based on TTL settings")
else:
    print("ℹ️  Skipping cleanup - demo data will remain in Redis")
    print("   💡 Data will automatically expire based on TTL settings")

## 🎯 Summary

Congratulations! You've successfully:

### ✅ **Completed Tasks:**
- **Configured** Azure Redis with access key authentication
- **Connected** NexaMem to Azure Cache for Redis
- **Stored and retrieved** conversation history
- **Tested** multiple conversation contexts
- **Verified** connection health and performance

### 🔐 **Access Key Authentication Benefits:**
- **Simple setup**: Just hostname + access key
- **Quick start**: No identity configuration needed
- **Universal**: Works with any Redis client
- **Development-friendly**: Perfect for testing and POCs

### 🚀 **Next Steps:**

1. **Integrate into your application**: Use the patterns from this notebook
2. **Configure TTL and quotas**: Set appropriate data retention policies
3. **Monitor usage**: Track memory consumption and performance
4. **Consider enterprise features**: Check out [quickstart_azure_redis_enterprise.ipynb](quickstart_azure_redis_enterprise.ipynb) for:
   - Entra ID authentication
   - Azure Key Vault integration
   - Advanced security features
   - Production-ready patterns

### 📚 **Resources:**
- [NexaMem Documentation](https://github.com/your-repo/nexamem)
- [Azure Cache for Redis Documentation](https://docs.microsoft.com/en-us/azure/azure-cache-for-redis/)
- [Redis Python Client Documentation](https://redis-py.readthedocs.io/)

---

**Happy coding with NexaMem and Azure Redis! 🚀**